In [ ]:
# population 01.01.2024: tps00001
# https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp/default/table?lang=en

# Gross national income (GNI) per capita: https://ec.europa.eu/eurostat/databrowser/view/nama_10_pp/default/table?lang=en

# asylum is the number of asylum applicants: https://ec.europa.eu/eurostat/databrowser/view/migr_asyappctza/default/table?lang=en

# protection is the number of first instance positive decisions on applications : https://ec.europa.eu/eurostat/databrowser/view/migr_asydcfsta/default/table?lang=en

# temporary is the number of beneficiaries of temporary protection: https://ec.europa.eu/eurostat/databrowser/view/migr_asytpsm__custom_17258546/default/table?lang=en

# returns is third country nationals returned following an order to leave: https://ec.europa.eu/eurostat/databrowser/view/migr_eirtn/default/table?lang=en

# visa is the number of total uniform visas applied for short stays : https://home-affairs.ec.europa.eu/policies/schengen/visa-policy/short-stay-visas-issued-schengen-countries_en
# Visa statistics for consulates

## 🇪🇺 EU Budget Allocation Breakdown (2028–2034 Proposal)

| Category                  | Amount (€ billion) |
|---------------------------|---------------------|
| **Total Allocation**      | **€832.9 billion**  |
| General Allocation        | €748.9 billion      |
| Migration & Security      | €34.3 billion       |
| Social Climate Fund (SCF) | €49.9 billion       |


In [ ]:
# Social Climate Fund (NRP regulation, page 37: 50 100 000 000)

# 🇪🇺 Bᵢ Component in EU Funding Allocation  
### *Migration, Borders, and Internal Security*

This formula calculates the **Bᵢ component**, used to allocate EU funds based on a country’s **border exposure**, **migration pressure**, and **structural vulnerability**.

---

## 🔷 What is Bᵢ?

It’s a **weighted score for country _i_**, combining:

- **Border pressure**
- **Migration statistics**
- **Security-related vulnerability**

Each component has its own subweights.

### 📌 Formula:

$$
B_i = 50\% \cdot \text{Borders} + 30\% \cdot \text{Migration} + 20\% \cdot \text{Security}
$$

---

## 📘 1. Borders (45%)

### Weighted components:

$$
\text{Borders} = 90\% \cdot (\text{sea borders} + \text{external land borders}) + 10\% \cdot \text{VISA applications}
$$

- **Sea borders**: Length of coastline (geodesic length)
- **External borders**: Borders with **non-EU countries**, weighted further:
  - **1.00×** for all other external borders  
  - **1.25×** for direct **land borders with Russia (RU) or Belarus (BY)** → higher strategic risk

$$
\text{Border score} = 90\% \cdot (\text{sea} + \text{external}_{\text{weighted}}) + 10\% \cdot \text{visa volume}
$$

---

## 🟩 2. Migration (35%)

This component is based on the **average value of four indicators** (Eurostat data):

$$
\text{Migration} = \text{average}(\text{asylum applicants}, \text{protection granted}, \text{temporary protection}, \text{returns})
$$

- Data is averaged over **2022–2024**, per regulation.
- Each country gets a **relative score** based on how many of these cases it processes.

---

## 🟦 3. Security (20%)

This combines **three structural factors**:

$$
\text{Security} = 0.4 \cdot \text{Population share} + 0.45 \cdot \text{Inverted GNI per capita (PPS) share} + 0.15 \cdot \text{Surface area share}
$$

- **Population share**: Country’s share of total EU population  
- **Inverted GNI/pc**: Countries with **lower income** get **higher weight**  
- **Surface share**: Larger countries are more costly to secure

✅ Countries that are **large, poor, and populous** receive **more support** here.

---

## 📊 Summary Table

| Component           | Subcomponents                                                     | Weights      |
|--------------------|--------------------------------------------------------------------|--------------|
| **Borders (50%)**  | 90%: Sea & external land borders (weighted), 10%: VISA applications| 90% + 10%    |
| **Migration (30%)**| Average of: asylum, protection, temporary protection, returns      | Equal weight |
| **Security (20%)** | 40%: Population, 45%: Inverted GNI/pc (PPS), 15%: Surface area      | 100% total   |

In [ ]:
import geopandas as gpd
import pandas as pd


def compute_geo_data(path):
    # https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/countries
    gdf = gpd.read_file(path)
    gdf = gdf[gdf["CNTR_ID"] != "AQ"].copy()  # remove Antarctica
    gdf = gdf.set_index("CNTR_ID")
    high_risk_neighbors = {"RU", "BY"}

    results = []

    for country_id, row in gdf[gdf["EU_STAT"] == "T"].iterrows():
        iso3 = row["ISO3_CODE"]
        name = row["CNTR_NAME"]
        geom = row["geometry"]
        boundary = geom.boundary

        total_land_border_km = 0.0
        external_border_km = 0.0
        external_high_risk_km = 0.0
        external_regular_km = 0.0
        sea_border_geom = boundary

        for neighbor_id, neighbor_row in gdf.drop(country_id).iterrows():
            neighbor_geom = neighbor_row["geometry"]
            shared_border = boundary.intersection(neighbor_geom)

            if shared_border.is_empty:
                continue

            length_km = shared_border.length / 1000

            if gdf.loc[neighbor_id, "EU_STAT"] != "T":
                # External (non-EU) neighbor
                external_border_km += length_km
                if neighbor_id in high_risk_neighbors:
                    external_high_risk_km += length_km
                else:
                    external_regular_km += length_km

            total_land_border_km += length_km
            sea_border_geom = sea_border_geom.difference(shared_border)

        sea_border_km = sea_border_geom.length / 1000
        surface_km2 = geom.area / 1_000_000

        results.append(
            {
                "geo_codes": country_id,
                "ISO3": iso3,
                "Name": name,
                "Land_border_km": round(total_land_border_km, 2),
                "Sea_border_km": round(sea_border_km, 2),
                "External_border_km": round(external_border_km, 2),
                "External_regular_km": round(external_regular_km, 2),
                "External_high_risk_km": round(external_high_risk_km, 2),
                "Surface_km2": round(surface_km2, 2),
            }
        )

    return pd.DataFrame(results)

In [ ]:
df_results_20 = compute_geo_data("input/CNTR_RG_20M_2024_3035.gpkg").assign(
    **{"resolution": "1:20M"}
)
df_results_10 = compute_geo_data("input/CNTR_RG_10M_2024_3035.gpkg").assign(
    **{"resolution": "1:10M"}
)
df_results_03 = compute_geo_data("input/CNTR_RG_03M_2024_3035.gpkg").assign(
    **{"resolution": "1:03M"}
)
df_results_01 = compute_geo_data("input/CNTR_RG_01M_2024_3035.gpkg").assign(
    **{"resolution": "1:01M"}
)

values_to_pivot = "Sea_and_external_land_border_wgt"
# values_to_pivot = "Land_border_km"
# values_to_pivot = "Sea_border_km"

df_res = pd.concat([df_results_20, df_results_10, df_results_03, df_results_01])

df_res["External_border_wgt"] = (
    1 * df_res["External_regular_km"] + 1.25 * df_res["External_high_risk_km"]
)
df_res["Sea_and_external_land_border_wgt"] = (
    df_res["Sea_border_km"] + df_res["External_border_wgt"]
)

df_res = df_res.pivot(
    index=["geo_codes", "Name"], values=values_to_pivot, columns="resolution"
).reset_index()

cols = ["1:01M", "1:03M", "1:10M", "1:20M"]
df_res[cols] = 100 * df_res[cols] / df_res[cols].sum()

In [ ]:
df_res

In [ ]:
database_country = pd.read_excel("input/database_country.xlsx")

In [ ]:
param_border = 0.45
param_migration = 0.35
param_security = 0.20

df_results = compute_geo_data("input/CNTR_RG_20M_2024_3035.gpkg")

df_results["External_border_wgt"] = (
    1 * df_results["External_regular_km"] + 1.25 * df_results["External_high_risk_km"]
)
df_results["Sea_and_external_land_border_wgt"] = (
    df_results["Sea_border_km"] + df_results["External_border_wgt"]
)

df_results = df_results.merge(
    database_country[
        [
            "geo_codes",
            "geo_labels",
            "population_2024",
            "gni_pc_pps_2024",
            "asylum_avg_2022_2024",
            "protection_avg_2022_2024",
            "temporary_sum_2022_2024",
            "returns_avg_2022_2024",
            "visas_avg_2022_2024",
        ]
    ],
    on="geo_codes",
    how="left",
)

df_results["Inv_GNI_share"] = (
    100
    * (1 / df_results["gni_pc_pps_2024"])
    / (1 / df_results["gni_pc_pps_2024"]).sum()
)
df_results["Surface share"] = 100 * (
    df_results["Surface_km2"] / df_results["Surface_km2"].sum()
)
df_results["Population share"] = 100 * (
    df_results["population_2024"] / df_results["population_2024"].sum()
)

df_results["Borders"] = 100 * (
    0.9
    * (
        df_results["Sea_and_external_land_border_wgt"]
        / df_results["Sea_and_external_land_border_wgt"].sum()
    )
    + 0.1 * df_results["visas_avg_2022_2024"] / df_results["visas_avg_2022_2024"].sum()
)

migration_cols = [
    "asylum_avg_2022_2024",
    "protection_avg_2022_2024",
    "temporary_sum_2022_2024",
    "returns_avg_2022_2024",
]

shares_df = (
    df_results[migration_cols].div(df_results[migration_cols].sum(axis=0), axis=1) * 100
)

df_results["Migration"] = shares_df.mean(axis=1)

df_results["Security"] = (
    0.4 * df_results["Population share"]
    + 0.45 * df_results["Inv_GNI_share"]
    + 0.15 * df_results["Surface share"]
)

df_results["B"] = (
    param_border * df_results["Borders"]
    + param_migration * df_results["Migration"]
    + param_security * df_results["Security"]
)

B_amount_bn = 34.215

df_results["B_alloc_bn"] = B_amount_bn * 0.01 * df_results["B"]

In [ ]:
df_results[
    ["geo_codes", "geo_labels", "B_alloc_bn", "B", "Borders", "Migration", "Security"]
]

In [ ]:
if df_results.notna().all().all():
    print("Nincs üres érték!")

In [ ]:
# https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/countries